# __RAG__

We will see why RAG exists in the first place. LLMs are compressed, lossy representations of their training data. Foundational models cannot access information beyond the date their training data was collected. This leads to some fundamental failures:
1. __Static Knowledge Cutoffs:__ The model's weights are frozen in time, rendering it incapable of answering queries about recent events.
2. __Model Capacity Limits:__ An LLM acts as a lossy compression algorithm over its training corpus, meaning it cannot memorize high-cardinality, niche, or exact private data.
3. __Lack of Access to Private Data:__ Foundational models are trained on public digital data and do not possess enterprise confidatial or personalized data.

When the LLM is forced to sample from a distribution where the probability mass is highly uncertain (the model does not know the answer), the model generates outputs that are factually incorrect or lack grounding. This is a __hallucination__.

To solve this problem, we inject deterministic, retrieved context into the stochastic generation process. 

### __Definition of RAG__
In autoregressive generation, the model predicts the next token based on the user prompt and the previously generated tokens. RAG alters this fundamental idea by introducing a non-parametric memory variable that represents a set of retrieved documents. The generative distribution is now strictly conditioned on both the prompt and the retrieved context.

The non-parametric variable is obtained through _Maximum Inner Product Search (MIPS)_ or _Cosine Similarity_ in a high-dimensional vector space. Embedding models convert the meaning of words and sentences into multidimensional vectors. Looking at this from a system perspective, RAG introduces an I/O and network bottleneck. We are trading GPU compute time for database latency, impacting the Time-To-First-Token (TTFT).

### __Architecture Flow__
A RAG system requires a very well architected flow of data.
1. __User Query:__ Its the raw string input that arrives at the API.
2. __Query Rewriting:__ The raw query can be very ambiguous. Because of that, we can use an LLM or heuristic to rewrite the query into an optimal search string.
3. __Embedding Model:__ The rewritten query is passed through an encoder to generate a dense vector representation.
4. __Vector/Hybrid Search:__ The query vector is compared against a pre-computed index of document chunks. To ensure scalability, databases partition this data and use Approximate Nearest Neighbor (ANN) algorithms like _HNSW_.
5. __Context Augmentation:__ The top K documents are retrieved. We apply a reranking model (like Cross-Encoder) to sort the documents by relevance.
6. __LLM Inference:__ The context is concatenated with the prompt. The LLM processes this massive context window to generated the final response.
7. __Output Guardrails:__ The generated output is validated to ensure no restricted data is leaked and that the answer is strictly derived from the provided context.

### __Types of RAG__